# SpaceX Falcon 9 Landing Success Prediction: Final Summary

This notebook summarizes the completed portfolio version of the SpaceX Falcon 9 first-stage landing prediction project. It preserves the original course notebooks and presents the final analysis, figures, and model results in a concise recruiter-facing format.

**Dataset scope:** 90 launches from 2010-06-04 through 2020-11-05.

**Target:** `Class`, where `1` means the first stage landed successfully and `0` means it did not.

**Overall landing success rate:** 66.7%.

## Business Problem

Reusable first-stage boosters can materially reduce launch cost. A model that estimates landing success helps frame launch risk, compare mission profiles, and identify operational factors associated with successful recovery.

## Project Workflow

1. Collect Falcon 9 launch records from public sources.
2. Clean fields and derive a binary landing outcome.
3. Explore success patterns by year, launch site, orbit, payload, and booster characteristics.
4. Build classification models using one-hot encoded launch features.
5. Evaluate models on a held-out test set and document limitations.

## Data Assets

The final artifacts use local CSV snapshots in `data/processed/` so the project can be reviewed without re-downloading data:

- `dataset_part_1.csv`: launch records after initial collection and cleaning.
- `dataset_part_2.csv`: launch records with the binary `Class` outcome.
- `dataset_part_3.csv`: one-hot encoded feature matrix used for machine learning.

In [1]:
from pathlib import Path
import pandas as pd

root = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
data = pd.read_csv(root / 'data' / 'processed' / 'dataset_part_2.csv')
features = pd.read_csv(root / 'data' / 'processed' / 'dataset_part_3.csv')

print(f'Launch rows: {len(data)}')
print(f'Feature columns: {features.shape[1]}')
data[['FlightNumber', 'Date', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Class']].head()

Launch rows: 90
Feature columns: 83


,FlightNumber,Date,PayloadMass,Orbit,LaunchSite,Outcome,Class
0,1,2010-06-04,6104.959412,LEO,CCAFS SLC 40,None None,0
1,2,2012-05-22,525.000000,LEO,CCAFS SLC 40,None None,0
2,3,2013-03-01,677.000000,ISS,CCAFS SLC 40,None None,0
3,4,2013-09-29,500.000000,PO,VAFB SLC 4E,False Ocean,0
4,5,2013-12-03,3170.000000,GTO,CCAFS SLC 40,None None,0


## Exploratory Findings

Landing success improves over time in the dataset, which is consistent with Falcon 9 booster reuse and recovery maturing over successive flights. Launch site and orbit also show visible differences, although several orbit categories have very small sample sizes and should not be over-interpreted.

![Landing success trend by year](../figures/landing_success_trend_by_year.png)

![Success rate by launch site](../figures/success_rate_by_launch_site.png)

![Success rate by orbit](../figures/success_rate_by_orbit.png)

![Payload mass by outcome](../figures/payload_mass_by_outcome.png)

## Model Results

The final comparison uses a train/test split with stratification and fits preprocessing only on the training data. This avoids using the held-out test rows during model selection.

| Model | Best CV Accuracy | Test Accuracy | Best Parameters |
|---|---:|---:|---|
| Logistic Regression | 0.835 | 0.833 | `{'model__C': 0.1, 'model__penalty': 'l1'}` |
| SVM | 0.877 | 0.833 | `{'model__C': 0.01, 'model__gamma': 'scale', 'model__kernel': 'linear'}` |
| KNN | 0.864 | 0.833 | `{'model__n_neighbors': 9, 'model__p': 1, 'model__weights': 'distance'}` |
| Decision Tree | 0.863 | 0.778 | `{'criterion': 'gini', 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}` |

Multiple models tied on held-out accuracy. Logistic regression is selected as the primary model because it achieved the same test accuracy as the top group while remaining more interpretable.

**Selected model:** Logistic Regression

**Held-out test accuracy:** 83.3%

**Best cross-validation accuracy:** 83.5%

**Best parameters:** `{'model__C': 0.1, 'model__penalty': 'l1'}`

![Model comparison accuracy](../figures/model_comparison_accuracy.png)

## Error Profile

The selected model's confusion matrix uses labels `[0, 1]`, where `0` means did not land and `1` means landed. On the 18-row test set, the model correctly classified 15 launches and misclassified 3.

- Did-not-land recall: 50.0%
- Landed recall: 100.0%
- Weighted F1-score: 81.5%

![Confusion matrix for selected model](../figures/confusion_matrix_best_model.png)

## Interpretability

The coefficient plot shows the largest standardized logistic regression coefficients by absolute value. Positive coefficients push the prediction toward successful landing, while negative coefficients push it toward unsuccessful landing. Because the dataset is small and includes one-hot encoded categories, these coefficients should be read as directional model signals, not causal effects.

![Top logistic regression coefficients](../figures/logistic_regression_top_coefficients.png)

## Limitations

- The dataset has only 90 launches, so the 18-row test set is small.
- Several orbit categories have only one or a few examples.
- The project predicts historical landing outcomes and should not be treated as a production launch-risk model.
- Additional mission, weather, booster, and telemetry features would be needed for a more robust operational model.

## Conclusion

The project demonstrates a full applied data science workflow: data collection, cleaning, exploratory analysis, geospatial context, classification modeling, and business interpretation. In the leakage-safe final evaluation, logistic regression, SVM, and KNN reached 83.3% accuracy on the held-out test set, with logistic regression selected for its interpretability.